In [1]:
import cv2
import numpy as np

points = []

def mouse_callback(event, x, y, flags, param):
    global points, img_display
    if event == cv2.EVENT_LBUTTONDOWN:
        if len(points) < 4:
            points.append((x, y))
            print(f"Point {len(points)}: ({x}, {y})")
            cv2.circle(img_display, (x, y), 5, (0, 0, 255), -1)
            cv2.imshow("Image", img_display)
            
        if len(points) == 4:
            print("\nFour points selected:")
            for i, p in enumerate(points):
                print(f"P{i+1}: {p}")
            print("Press any key to superimpose the flag.")

# --- MAIN EXECUTION (Outside the function) ---

# 1. Load the background turf
img = cv2.imread("media\\turf.jpg")
# 2. Load the flag to superimpose
flag = cv2.imread("media\\srilanka.png") 

if img is None or flag is None:
    print("Error: Images not found. Check media folder paths.")
    exit()

img_display = img.copy()

cv2.namedWindow("Image")
cv2.setMouseCallback("Image", mouse_callback)
cv2.imshow("Image", img_display)

# This loop stays open until you press a key
cv2.waitKey(0)
cv2.destroyAllWindows()

# Convert list to numpy array for calculation
points_arr = np.array(points, dtype=np.float32)

if len(points) == 4:
    # Source: Corners of the flag image (TL, TR, BR, BL)
    h_f, w_f = flag.shape[:2]
    pts_src = np.array([[0, 0], [w_f - 1, 0], [w_f - 1, h_f - 1], [0, h_f - 1]], dtype=np.float32)
    
    # Destination: The points you clicked on the turf
    pts_dst = points_arr
    
    # Calculate Homography Matrix
    H, status = cv2.findHomography(pts_src, pts_dst)
    
    # Warp the flag image into the perspective of the turf
    warped_flag = cv2.warpPerspective(flag, H, (img.shape[1], img.shape[0]))
    
    # Create a mask for the area where the flag will go
    mask = np.zeros_like(img, dtype=np.uint8)
    cv2.fillConvexPoly(mask, pts_dst.astype(int), (255, 255, 255))
    
    # Combine the warped flag with the original turf
    # (Where mask is white, use warped_flag; otherwise keep original turf)
    img_final = np.where(mask == 255, warped_flag, img)
    
    print("\nFinal array of selected points:")
    print(points_arr)
    
    cv2.imshow("Final Result", img_final)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

Point 1: (688, 166)
Point 2: (844, 159)
Point 3: (1249, 555)
Point 4: (225, 549)

Four points selected:
P1: (688, 166)
P2: (844, 159)
P3: (1249, 555)
P4: (225, 549)
Press any key to superimpose the flag.

Final array of selected points:
[[ 688.  166.]
 [ 844.  159.]
 [1249.  555.]
 [ 225.  549.]]
